<a href="https://colab.research.google.com/github/mulata12/NLLB-Tig-Eng-healthcare-mt/blob/main/notebooks/01_baseline_nllb_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!git clone https://github.com/mulata12/NLLB-Tig-Eng-healthcare-mt.git

In [10]:
%cd NLLB-Tig-Eng-healthcare-mt
!ls

[Errno 2] No such file or directory: 'NLLB-Tig-Eng-healthcare-mt'
/content/NLLB-Tig-Eng-healthcare-mt
configs  docs	 notebooks  requirements.txt  src
data	 models  README.md  results


In [8]:
%%capture
!pip install -r requirements.txt

In [11]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [13]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
       

In [14]:
src_lang = "eng_Latn"
tgt_lang = "tir_Ethi"

In [20]:
def translate(text, src_lang, tgt_lang):

    tokenizer.src_lang = src_lang

    inputs = tokenizer(text, return_tensors="pt").to(device)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=200
    )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [16]:
sentences = [
    "Take the medicine twice daily.",
    "Consult a doctor immediately.",
    "Drink plenty of water.",
    "The patient has a fever.",
    "Wash your hands before eating."
]

In [21]:
results = []

for sentence in sentences:

    translation = translate(sentence, src_lang, tgt_lang)

    print("English:", sentence)
    print("Tigrigna:", translation)
    print()

    results.append({
        "English": sentence,
        "Tigrigna_translation": translation
    })

English: Take the medicine twice daily.
Tigrigna: ነቲ መድሃኒት መዓልታዊ ክልተ ግዜ ይውሰዶ።

English: Consult a doctor immediately.
Tigrigna: ብቕልጡፍ ናብ ሓኪም ተወከስ።

English: Drink plenty of water.
Tigrigna: ብዙሕ ማይ ጠዓዮ።

English: The patient has a fever.
Tigrigna: እቲ ሕሙም ትኩሳት ኣለዎ።

English: Wash your hands before eating.
Tigrigna: ቅድሚ ምግቢ ኣእዳውካ ሓጸብ።



In [23]:
#!ls

configs  docs	 notebooks  requirements.txt  src
data	 models  README.md  results


In [24]:
#!ls results

metrics  mqm_annotations  translations


In [28]:
import pandas as pd

df = pd.DataFrame(results)

df.head()

,English,Tigrigna_translation
0,Take the medicine twice daily.,ነቲ መድሃኒት መዓልታዊ ክልተ ግዜ ይውሰዶ።
1,Consult a doctor immediately.,ብቕልጡፍ ናብ ሓኪም ተወከስ።
2,Drink plenty of water.,ብዙሕ ማይ ጠዓዮ።
3,The patient has a fever.,እቲ ሕሙም ትኩሳት ኣለዎ።
4,Wash your hands before eating.,ቅድሚ ምግቢ ኣእዳውካ ሓጸብ።


In [29]:
df.to_csv("results/translations/baseline_en_ti.csv", index=False)

print("Baseline results saved successfully.")

Baseline results saved successfully.


In [30]:
#!ls results/translations

baseline_en_ti.csv
